# bob, explained - Episode 5: BRAM

BRAM

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep05BRAM
# =============================================================================
#  bob, explained - EPISODE 5: BRAM
#  BRAM
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E05S1What
#      E05S2Cfg
#      E05S3Modes
#      E05S4Contents
#      E05S5Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 5 - BRAM: a UG473 RAMB18E1 subset, and why contents are separate
# =============================================================================

def s1_what(sc):
    sc.heading("bob's BRAM is a real RAMB18",
               "hw/src/tiles/bram_core.v is written to Vivado's inference template on purpose")

    mem = Rectangle(width=3.4, height=2.6, color=C_VPR, stroke_width=3)
    mem.set_fill(C_VPR, opacity=0.12)
    t = Text("1024 x 18\ntrue dual port", font_size=24, color=INK, line_spacing=0.8)
    core = VGroup(mem, t.move_to(mem)).move_to(np.array([0, 0.9, 0]))
    sc.play(FadeIn(core), run_time=0.7)

    for side, x, col in (("port A", -3.6, C_RTL), ("port B", 3.6, C_GRF)):
        pins = code_block(["addr[9:0]", "di[17:0]", "we", "en", "rst", "regce",
                           "", "do[17:0]"], 17, col)
        pins.move_to(np.array([x, 0.9, 0]))
        sc.play(FadeIn(pins), run_time=0.6)
        sc.play(GrowArrow(arrow(pins.get_right() if x < 0 else pins.get_left(),
                                mem.get_left() if x < 0 else mem.get_right(),
                                DIM, 0.25)), run_time=0.3)

    facts = code_block([
        "one block  ->  one RAMB18E1 on the host FPGA, not a pile of LUTs",
        "synchronous read (UG473): the address is registered, data comes next cycle",
        "bob has two of them, in grid column x = 3, each 5 rows tall",
    ], 19, INK)
    facts.next_to(core, DOWN, buff=1.2).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in facts], lag_ratio=0.2), run_time=1.6)

    why = Text("A guest FPGA whose memories melt into the host's LUTs would not fit. "
               "Inference is the whole reason this block is written the way it is.",
               font_size=19, color=C_BIT)
    why.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.0)


def s2_cfg(sc):
    sc.heading("Eight configuration bits", "that is the entire BRAM tile - the data is somewhere else")

    fields = [("wmode_a", 2, C_RTL), ("wmode_b", 2, C_GRF), ("reg_a", 1, C_BIT),
              ("reg_b", 1, C_BIT), ("jtag_a", 1, C_PY), ("jtag_b", 1, C_PY)]
    bar = fieldbar(fields, total_w=8.0, h=0.8, size=17)
    bar.shift(UP * 1.5)
    sc.play(Create(bar[0]), FadeIn(bar[1]), FadeIn(bar[2]), run_time=1.2)

    rows = [
        ("wmode_a / wmode_b", "0 WRITE_FIRST, 1 READ_FIRST, 2 NO_CHANGE  (UG473)"),
        ("reg_a / reg_b", "DOA_REG / DOB_REG - one more pipeline stage on the output"),
        ("jtag_a / jtag_b", "let USER4 drive that port's pins, so write modes can be"),
        ("", "stepped one clock at a time from the host"),
    ]
    g = VGroup()
    for a, b in rows:
        g.add(VGroup(mono(a, 19, C_BIT), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 12.8:
        g.scale_to_fit_width(12.8)
    g.next_to(bar, DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in g], lag_ratio=0.18),
            run_time=1.8)

    trim = code_block([
        "trimmed from a real RAMB18E1, and we say so:",
        "   the 9 / 4 / 1-bit width modes   (bit-level addressing stops inference)",
        "   per-byte write enables",
        "   separate RSTRAM / RSTREG pins",
    ], 18, DIM)
    trim[0].set_color(C_ERR)
    trim.to_edge(DOWN, buff=0.35).set_x(0)
    sc.play(FadeIn(trim), run_time=0.9)
    sc.wait(2.2)


def s3_modes(sc):
    sc.heading("Three write modes out of one template",
               "only READ_FIRST is inferable - the other two are rebuilt around it")

    base = chip("bram_core.v\nVivado's READ_FIRST template", C_VPR, 5.0, 1.2, 20)
    base.shift(UP * 1.8)
    sc.play(FadeIn(base), run_time=0.6)

    outs = VGroup(
        chip("READ_FIRST\nthe old word appears", C_RTL, 3.4, 1.1, 18),
        chip("WRITE_FIRST\nthe new word appears", C_BIT, 3.4, 1.1, 18),
        chip("NO_CHANGE\nthe output holds", C_GRF, 3.4, 1.1, 18),
    ).arrange(RIGHT, buff=0.5).next_to(base, DOWN, buff=1.1)
    for o in outs:
        sc.play(GrowArrow(arrow(base.get_bottom(), o.get_top(), DIM, 0.12)),
                FadeIn(o), run_time=0.45)

    how = code_block([
        "WRITE_FIRST and NO_CHANGE are built from READ_FIRST with a register and a",
        "mux on the output path - which is exactly why the write mode can be a",
        "CONFIGURATION BIT rather than a synthesis-time parameter.",
        "",
        "A real FPGA picks the mode when you instantiate the primitive.",
        "bob picks it when you load a bitstream.",
    ], 19, INK)
    how[4].set_color(DIM)
    how[5].set_color(C_BIT)
    how.next_to(outs, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in how], lag_ratio=0.18), run_time=2.0)

    proof = Text("tb_bram.v sweeps all 36 mode x register x port combinations "
                 "against model.py: 5292 checks.", font_size=19, color=C_BIT)
    proof.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(proof), run_time=0.8)
    sc.wait(2.2)


def s4_contents(sc):
    sc.heading("Contents are not configuration",
               "UG470 keeps BRAM data in its own block type, and so does bob - with two ways in")

    mem = chip("the 1024 words", C_VPR, 3.2, 1.0, 21).shift(UP * 1.9)
    sc.play(FadeIn(mem), run_time=0.5)

    a = chip("USER4\nSELECT, LOAD_PTR,\nWRITE, READ", C_PY, 3.4, 1.4, 18)
    b = chip("FAR block type 001\nframes, inside the\nsame CRC stream", C_BIT, 3.8, 1.4, 18)
    a.move_to(np.array([-3.4, -0.1, 0]))
    b.move_to(np.array([3.4, -0.1, 0]))
    sc.play(GrowArrow(arrow(a.get_top(), mem.get_bottom(), DIM, 0.12)), FadeIn(a),
            run_time=0.6)
    sc.play(GrowArrow(arrow(b.get_top(), mem.get_bottom(), DIM, 0.12)), FadeIn(b),
            run_time=0.6)
    al = mono("M5, still used by --mode chain", 15, DIM).next_to(a, DOWN, buff=0.18)
    bl = mono("M15, the default", 15, DIM).next_to(b, DOWN, buff=0.18)
    sc.play(FadeIn(al), FadeIn(bl), run_time=0.4)

    frame = code_block([
        "block type 001:  column = BRAM index,  frame n = row x 128 + minor",
        "frame n holds addresses 4n .. 4n+3,  word = { 14'b0, data[17:0] }",
        "256 frames per BRAM, and FAR auto-increments into the next one",
    ], 18, C_BIT)
    frame.next_to(VGroup(a, b), DOWN, buff=0.8).set_x(0)
    sc.play(FadeIn(frame), run_time=0.9)

    rule = code_block([
        "both paths write only while GWE = 0 - you cannot rewrite a running memory",
        "JPROGRAM does NOT clear contents: they survive a reprogram",
    ], 19, C_ERR)
    rule.next_to(frame, DOWN, buff=0.55).set_x(0)
    sc.play(FadeIn(rule), run_time=0.8)
    sc.wait(1.2)

    bug = Text("M11's bug, found on the board: bob load wrote only up to the last non-zero "
               "word, so zero words kept the PREVIOUS design's values. "
               "Now all 1024 words of every used BRAM are always written.",
               font_size=18, color=C_ERR)
    bug.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(bug), run_time=0.9)
    sc.wait(2.4)


def s5_files(sc):
    sc.files_used(
        inputs=[("hw/src/tiles/bram_core.v", "the inferable 1024x18 TDP memory"),
                ("hw/src/tiles/bram_block.v", "fabric pins or the USER4 drive word"),
                ("hw/src/tiles/bram_jtag.v", "USER4 commands + the M15 frame sequencer")],
        generated=[("tools/bob/device.json", "the 8 fields, USER4 command codes"),
                   ("tools/bob/model.py", "class Bram - the reference behaviour")],
        verified=[("hw/tb/tb_bram.v", "5292 checks, all 36 combinations"),
                  ("sim/mutate_fabric.sh", "bram-no-write-first, bram-en-ignored, ..."),
                  ("docs/hwtest/results.log", "ram-readback on the real board")])


EP05 = [s1_what, s2_cfg, s3_modes, s4_contents, s5_files]


class Ep05BRAM(BobScene):
    def construct(self):
        self.titlecard("EPISODE 5", "BRAM",
                       "a UG473 subset, and why contents are not configuration")
        for i, part in enumerate(EP05):
            part(self)
            if i < len(EP05) - 1:
                clear_all(self)


class E05S1What(BobScene):
    def construct(self): s1_what(self)


class E05S2Cfg(BobScene):
    def construct(self): s2_cfg(self)


class E05S3Modes(BobScene):
    def construct(self): s3_modes(self)


class E05S4Contents(BobScene):
    def construct(self): s4_contents(self)


class E05S5Files(BobScene):
    def construct(self): s5_files(self)